In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
from utils.metrics import evaluate_model, plot_confusion_matrix, plot_roc_curve

Load the trained model and test data

In [ ]:
from tensorflow.keras.models import load_model
from utils.data_loader import create_train_val_loaders

# Load the best model
best_model = load_model('models/best_model.h5')

# Load test data
test_loader = create_train_val_loaders(
    train_csv='data/processed/train/annotations.csv',
    val_csv='data/processed/test/annotations.csv',
    batch_size=32,
    image_size=(224, 224),
    max_frames=30
)

Make predictions on the test set

In [ ]:
y_true = []
y_pred = []
y_pred_proba = []

for X_batch, y_batch in test_loader:
    y_true.extend(np.argmax(y_batch, axis=-1))
    y_pred_batch = best_model.predict(X_batch)
    y_pred.extend(np.argmax(y_pred_batch, axis=-1))
    y_pred_proba.extend(y_pred_batch)

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_pred_proba = np.array(y_pred_proba)

Evaluate the model

In [ ]:
evaluation_results = evaluate_model(y_true, y_pred, y_pred_proba)

print("Evaluation Results:")
for metric, value in evaluation_results.items():
    if isinstance(value, dict):
        print(f"\n{metric}:")
        for sub_metric, sub_value in value.items():
            print(f"  {sub_metric}: {sub_value}")
    else:
        print(f"{metric}: {value}")

# Plot confusion matrix
plot_confusion_matrix(evaluation_results['confusion_matrix'], ['Real', 'Fake'])

# Plot ROC curve
plot_roc_curve(y_true, y_pred_proba)

Compare different models (if applicable)

In [ ]:
# Assuming you have multiple models to compare
models = ['model1', 'model2', 'model3']
accuracies = []
precisions = []
recalls = []
f1_scores = []
auc_rocs = []

for model_name in models:
    model = load_model(f'models/{model_name}_best.h5')
    y_pred = model.predict(test_loader)
    y_pred_class = np.argmax(y_pred, axis=-1)
    
    accuracies.append(accuracy_score(y_true, y_pred_class))
    precisions.append(precision_score(y_true, y_pred_class))
    recalls.append(recall_score(y_true, y_pred_class))
    f1_scores.append(f1_score(y_true, y_pred_class))
    auc_rocs.append(roc_auc_score(y_true, y_pred[:, 1]))

# Plot comparison bar charts
fig, axes = plt.subplots(1, 5, figsize=(25, 6))

axes[0].bar(models, accuracies)
axes[0].set_title('Accuracy')

axes[1].bar(models, precisions)
axes[1].set_title('Precision')

axes[2].bar(models, recalls)
axes[2].set_title('Recall')

axes[3].bar(models, f1_scores)
axes[3].set_title('F1 Score')

axes[4].bar(models, auc_rocs)
axes[4].set_title('AUC-ROC')

plt.tight_layout()
plt.show()

In [ ]:
Analyze misclassified samples

In [ ]:
misclassified_indices = np.where(y_true != y_pred)[0]
misclassified_videos = test_annotations.iloc[misclassified_indices]['video_path']

print("Misclassified Videos:")
for video_path in misclassified_videos[:10]:  # Show first 10 misclassified videos
    print(video_path)